# Team total — halves and quarters: points per team per period

Targets: each team's points in 1H, 2H (incl. OT), Q1, Q2, Q3 and Q4 (regulation). The
same walk-forward CV (2019–2023) as the full-game model. Weather-shortened games are
excluded.

No period betting lines yet (issue #4), so this notebook measures **accuracy only**.
Spread and total for each period derive from the two team predictions, like the full game.

Model code: `src/canes_cfb/periods.py`. Weekly predictions include every period
(`scripts/predict_week.py`).

In [ ]:
import json

import lightgbm as lgb
import numpy as np
import pandas as pd

from canes_cfb.modeling import FEATURES
from canes_cfb.paths import PROCESSED, ROOT
from canes_cfb.periods import (
    PERIODS,
    add_period_shares,
    add_period_targets,
    fit_predict_period,
    period_features,
)
from canes_cfb.validation import walk_forward

features = pd.read_parquet(PROCESSED / "team_games.parquet")
features = add_period_shares(add_period_targets(features))
lgb_params = json.loads((ROOT / "models" / "team_points_params.json").read_text())["lightgbm"][
    "params"
]
data = features[features.completed & ~features.shortened & features.season.between(2016, 2023)]
data = data.reset_index(drop=True)

## 1. Baselines vs. model (walk-forward CV MAE, points per team)

- **league avg**: the constant average for the period
- **exp × league share**: the ratings' expected game points × the league's share for the period
- **exp × team share**: same, with each team's own as-of share (fast starters, strong closers)
- **lightgbm**: league-share base + LightGBM on the residual (the final model)

In [ ]:
rows = []
for p in PERIODS:
    y = data[f"pts_{p.value}"].to_numpy()
    errors = {"league avg": [], "exp × league share": [], "exp × team share": [], "lightgbm": []}
    for _, train, valid in walk_forward(data):
        share = y[train].sum() / data.points.to_numpy()[train].sum()
        league_base = data.exp_points.to_numpy() * share
        team_base = np.where(
            data[f"exp_pts_{p.value}"].isna(), league_base, data[f"exp_pts_{p.value}"]
        )
        errors["league avg"].append(np.abs(y[valid] - y[train].mean()).mean())
        errors["exp × league share"].append(np.abs(y[valid] - league_base[valid]).mean())
        errors["exp × team share"].append(np.abs(y[valid] - team_base[valid]).mean())
        pred = fit_predict_period(p, data.iloc[train], data.iloc[valid], FEATURES, lgb_params)
        errors["lightgbm"].append(np.abs(y[valid] - pred).mean())
    rows.append(
        {"period": p.value, "mean pts": y.mean(), **{k: np.mean(v) for k, v in errors.items()}}
    )
table = pd.DataFrame(rows).set_index("period")
table["model vs constant %"] = 100 * (table["lightgbm"] / table["league avg"] - 1)
table.round(3)

- The model beats the constant in every period: ~12% in 1H, 7–8% in 2H and Q2,
  ~5% in Q1 and Q3, and **almost nothing in Q4**.
- **Team-specific scoring shares are worse than the league share.** How a team splits
  its points across quarters is mostly noise; it regresses to the league pattern.
- **Q4 is different**: expected points × share is *worse* than the constant. See below.

## 2. Game script: why Q4 doesn't follow team strength

In [ ]:
recent = data[data.season.between(2019, 2023)]
recent = recent.assign(
    expected_margin=pd.cut(recent.exp_margin, [-60, -17, -10, -3, 3, 10, 17, 60])
)
script = recent.groupby("expected_margin", observed=True).agg(
    n=("points", "size"),
    exp_points=("exp_points", "mean"),
    Q1=("pts_Q1", "mean"),
    Q2=("pts_Q2", "mean"),
    Q3=("pts_Q3", "mean"),
    Q4=("pts_Q4", "mean"),
)
script["Q4 share %"] = 100 * script.Q4 / script[["Q1", "Q2", "Q3", "Q4"]].sum(axis=1)
script.round(2)

Big favorites (expected to win by 17+) score only ~21% of their points in Q4. They
run the clock with the game decided. Big underdogs score ~30% in Q4 (garbage time). Q4
points barely grow with team strength, unlike Q1–Q3. That's a nonlinear effect of the
expected margin. Trees capture it (an additive model with the expected margin as a
feature gets the same result), so no hand-built interaction is needed.

## 3. Medians, not means

Models train with an L1 (absolute error) objective, so each prediction is the period's
**median**. That's the right number for an over/under (the 50/50 point), but medians
don't add up: Q1+Q2+Q3+Q4 medians sum below the full-game median, because quarters
have many zeros and a right-skewed distribution. Don't expect the periods to reconcile
exactly.

In [ ]:
sample = data[data.season == 2023].copy()
train = data[data.season < 2023]
for p in PERIODS:
    sample[f"pred_{p.value}"] = fit_predict_period(p, train, sample, FEATURES, lgb_params)
quarters = sample[[f"pred_Q{i}" for i in range(1, 5)]].sum(axis=1)
halves = sample.pred_1H + sample.pred_2H
pd.Series(
    {
        "actual mean points": sample.points.mean(),
        "sum of quarter predictions": quarters.mean(),
        "sum of half predictions": halves.mean(),
        "exp_points (full-game ratings)": sample.exp_points.mean(),
    }
).round(2)

## 4. Next

- Period lines (issue #4): when available, grade 1H/2H/quarter spreads and totals the
  same way as the full game (edge vs opener, CLV).
- Q4 is barely predictable beyond its average; expect little value there.